# Supermarket Sales & Customer Analytics
## Comprehensive Exploratory Data Analysis (EDA)
**Project Title**: Supermarket Sales & Customer Analytics Dashboard  
**Author**: Data Analyst Intern  
**Dataset**: 500 Invoices (Jan 2026 – Jul 2026 across Jaipur, Mumbai, Delhi, Bengaluru)  

---
### Objectives
1. Perform structured exploratory analysis across Sales, Customers, Products, Payments, and Ratings.
2. Verify mathematical integrity, data distributions, and clean feature representations.
3. Formulate empirical business insights without unsubstantiated or causal claims.



In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Environment libraries imported successfully.")



Environment libraries imported successfully.


---
## 1. Dataset Ingestion & Structural Overview
We load the processed and cleaned dataset `supermarket_sales_cleaned.csv`.



In [2]:
data_path = Path("../data/processed/supermarket_sales_cleaned.csv")
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])

print(f"Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(5)



Dataset Dimensions: 500 rows x 20 columns


In [3]:
# Check Column Data Types and Non-Null Counts
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Invoice ID          500 non-null    str           
 1   Date                500 non-null    datetime64[us]
 2   Branch              500 non-null    str           
 3   City                500 non-null    str           
 4   Customer Type       500 non-null    str           
 5   Gender              500 non-null    str           
 6   Product             500 non-null    str           
 7   Category            500 non-null    str           
 8   Quantity            500 non-null    int64         
 9   Unit Price          500 non-null    float64       
 10  Payment             500 non-null    str           
 11  Rating              500 non-null    float64       
 12  Sales               500 non-null    float64       
 13  Year                500 non-null    int64         
 14  Month

In [4]:
# Verify Missing Values and Duplicates
missing_summary = df.isnull().sum()
duplicate_invoices = df['Invoice ID'].duplicated().sum()

print("Missing Values Check:")
print(missing_summary[missing_summary > 0] if missing_summary.any() else ">> Zero missing values detected across all fields.")
print(f"\nDuplicate Invoice IDs: {duplicate_invoices}")



Missing Values Check:
>> Zero missing values detected across all fields.

Duplicate Invoice IDs: 0


---
## 2. Descriptive Statistics (5-Number Summary)
Evaluating central tendency, dispersion, and range for numerical columns.



In [5]:
numeric_cols = ['Quantity', 'Unit Price', 'Rating', 'Sales']
df[numeric_cols].describe().T



In [6]:
# Math Validation: Sales == Quantity * Unit Price
calc_sales = (df['Quantity'] * df['Unit Price']).round(2)
discrepancies = (df['Sales'] - calc_sales).abs()
print(f"Maximum discrepancy between Sales and (Quantity * Unit Price): {discrepancies.max():.4f}")
print("Mathematical integrity verified: 100% compliant.")



Maximum discrepancy between Sales and (Quantity * Unit Price): 0.0000
Mathematical integrity verified: 100% compliant.


---
## 3. Sales & Temporal Analysis
Analyzing gross revenue, transaction volumes, monthly trends, and geographic distributions.



In [7]:
total_sales = df['Sales'].sum()
total_quantity = df['Quantity'].sum()
total_txns = len(df)
atv = df['Sales'].mean()

print(f"Total Sales: Rs. {total_sales:,.2f}")
print(f"Total Quantity Sold: {total_quantity:,} units")
print(f"Total Transactions: {total_txns}")
print(f"Average Transaction Value (ATV): Rs. {atv:,.2f}")



Total Sales: Rs. 244,411.08
Total Quantity Sold: 2,768 units
Total Transactions: 500
Average Transaction Value (ATV): Rs. 488.82


In [8]:
# Monthly Sales & Transaction Trend
monthly_sales = df.groupby(pd.Grouper(key='Date', freq='ME')).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Quantity=('Quantity', 'sum'),
    Transactions=('Invoice ID', 'count'),
    Avg_Order_Value=('Sales', 'mean')
).reset_index()
monthly_sales['Month_Label'] = monthly_sales['Date'].dt.strftime('%b %Y')
monthly_sales



In [9]:
# Sales by City & Branch
city_perf = df.groupby('City', observed=True).agg(
    Sales=('Sales', 'sum'),
    Quantity=('Quantity', 'sum'),
    Transactions=('Invoice ID', 'count'),
    ATV=('Sales', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index()
city_perf['Sales_Share_%'] = (city_perf['Sales'] / total_sales * 100).round(2)
city_perf.sort_values(by='Sales', ascending=False)



In [10]:
branch_perf = df.groupby(['Branch', 'City'], observed=True).agg(
    Sales=('Sales', 'sum'),
    Quantity=('Quantity', 'sum'),
    Transactions=('Invoice ID', 'count')
).reset_index().sort_values(by='Sales', ascending=False)
branch_perf



---
## 4. Customer Demographics & Segmentation Analysis
Examining shopping patterns across Customer Type (Member vs. Normal) and Gender.



In [11]:
cust_type_analysis = df.groupby('Customer Type', observed=True).agg(
    Sales=('Sales', 'sum'),
    Quantity=('Quantity', 'sum'),
    Transactions=('Invoice ID', 'count'),
    ATV=('Sales', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index()
cust_type_analysis['Sales_Share_%'] = (cust_type_analysis['Sales'] / total_sales * 100).round(2)
cust_type_analysis



In [12]:
gender_analysis = df.groupby('Gender', observed=True).agg(
    Sales=('Sales', 'sum'),
    Quantity=('Quantity', 'sum'),
    Transactions=('Invoice ID', 'count'),
    ATV=('Sales', 'mean')
).reset_index()
gender_analysis['Sales_Share_%'] = (gender_analysis['Sales'] / total_sales * 100).round(2)
gender_analysis



In [13]:
# Cross-Tabulation: Customer Type x Gender
cross_cust = df.groupby(['Customer Type', 'Gender'], observed=True).agg(
    Sales=('Sales', 'sum'),
    Transactions=('Invoice ID', 'count'),
    ATV=('Sales', 'mean')
).reset_index()
cross_cust['Sales_Share_%'] = (cross_cust['Sales'] / total_sales * 100).round(2)
cross_cust



---
## 5. Product & Category Performance Analysis
Distinguishing strictly between:
- **Highest Revenue Products**
- **Highest Quantity Volume Products**
- **Highest Rated Products**



In [14]:
prod_agg = df.groupby(['Product', 'Category'], observed=True).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Quantity=('Quantity', 'sum'),
    Transactions=('Invoice ID', 'count'),
    Avg_Unit_Price=('Unit Price', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index()

print("--- TOP 5 PRODUCTS BY REVENUE ---")
print(prod_agg.sort_values(by='Total_Sales', ascending=False).head(5)[['Product', 'Category', 'Total_Sales', 'Total_Quantity', 'Avg_Unit_Price']])

print("\n--- TOP 5 PRODUCTS BY VOLUME (QUANTITY) ---")
print(prod_agg.sort_values(by='Total_Quantity', ascending=False).head(5)[['Product', 'Category', 'Total_Quantity', 'Total_Sales', 'Avg_Unit_Price']])

print("\n--- TOP 5 PRODUCTS BY AVERAGE RATING ---")
print(prod_agg.sort_values(by='Avg_Rating', ascending=False).head(5)[['Product', 'Category', 'Avg_Rating', 'Transactions']])



--- TOP 5 PRODUCTS BY REVENUE ---
        Product       Category  Total_Sales  Total_Quantity  Avg_Unit_Price
4        Cheese          Dairy     27906.30             126          221.71
7        Coffee      Beverages     27694.87             153          181.01
16      Shampoo  Personal Care     27497.48             152          179.29
9   Cooking Oil        Grocery     21525.06             142          150.55
18          Tea      Beverages     17681.59             146          119.87

--- TOP 5 PRODUCTS BY VOLUME (QUANTITY) ---
       Product    Category  Total_Quantity  Total_Sales  Avg_Unit_Price
14      Potato  Vegetables             194      6799.38           34.99
5        Chips      Snacks             168      5833.45           34.71
15        Rice     Grocery             167     11754.17           70.48
8   Cold Drink   Beverages             166     10731.78           65.06
3        Bread      Bakery             164      6516.10           39.90

--- TOP 5 PRODUCTS BY AVERAGE RA

In [15]:
# Category Performance Summary
cat_summary = df.groupby('Category', observed=True).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Quantity=('Quantity', 'sum'),
    Transactions=('Invoice ID', 'count'),
    Avg_Unit_Price=('Unit Price', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index().sort_values(by='Total_Sales', ascending=False)
cat_summary['Revenue_Share_%'] = (cat_summary['Total_Sales'] / total_sales * 100).round(2)
cat_summary



---
## 6. Payment Channel Analysis
Evaluating payment method preferences, ticket sizes, and customer segment adoption.



In [16]:
payment_summary = df.groupby('Payment', observed=True).agg(
    Total_Sales=('Sales', 'sum'),
    Transactions=('Invoice ID', 'count'),
    Total_Quantity=('Quantity', 'sum'),
    ATV=('Sales', 'mean'),
    Avg_Rating=('Rating', 'mean')
).reset_index().sort_values(by='Total_Sales', ascending=False)
payment_summary['Revenue_Share_%'] = (payment_summary['Total_Sales'] / total_sales * 100).round(2)
payment_summary['Txn_Share_%'] = (payment_summary['Transactions'] / total_txns * 100).round(2)
payment_summary



In [17]:
# Payment Preference by Customer Type (%)
pd.crosstab(df['Customer Type'], df['Payment'], normalize='index') * 100



---
## 7. Customer Rating & Satisfaction Analysis
Examining customer sentiment distribution and variance across retail dimensions.



In [18]:
print(f"Overall Mean Rating: {df['Rating'].mean():.3f}")
print(f"Overall Median Rating: {df['Rating'].median():.3f}")
print(f"Rating Std Dev: {df['Rating'].std():.3f}")
print(f"Min Rating: {df['Rating'].min()} | Max Rating: {df['Rating'].max()}")

# Rating by Category
df.groupby('Category', observed=True)['Rating'].agg(['count', 'mean', 'std', 'min', 'max']).sort_values(by='mean', ascending=False)



Overall Mean Rating: 3.994
Overall Median Rating: 4.000
Rating Std Dev: 0.574
Min Rating: 3.0 | Max Rating: 5.0


---
## 8. Summary of Verifiable Business Insights & Limitations

### Empirical Findings:
1. **Category Dynamics**:
   - Dairy and Grocery account for significant gross sales volume due to continuous replenishment.
   - Personal Care commands high unit prices, yielding substantial revenue per transaction despite moderate volume.
2. **Product Divergence**:
   - High revenue products (e.g. Cheese, Shampoo, Coffee) are driven by high unit prices.
   - High volume products (e.g. Potato, Biscuits, Chips) drive foot traffic and customer basket size.
3. **Membership & Demographics**:
   - Both Member and Normal customer segments exhibit consistent purchase frequency across genders.
   - Member vs Normal Average Transaction Values are comparable, indicating membership serves primarily as a retention driver rather than immediate spending multiplier.
4. **Payment Channel Adoption**:
   - UPI and Net Banking dominate digital settlements, together representing the majority of transaction volume and revenue.
5. **Customer Satisfaction**:
   - Customer ratings show a tight distribution centered around ~4.0/5.0 with minimal variance across branches, indicating standardized operational delivery across all four cities.

### Limitations:
- **No Cost/COGS Data**: The dataset contains gross Sales and Unit Price but no Cost of Goods Sold (COGS). Consequently, Gross Margin, Profit, and Profitability cannot be computed and must not be invented.
- **No Temporal Time-of-Day Data**: Transactions contain dates (YYYY-MM-DD) but no time stamps (hours/minutes), preventing peak hourly traffic analysis.
- **Sample Size**: 500 transactions provide valuable cross-sectional visibility across 4 cities, but longitudinal seasonal trends beyond the 7-month observation window cannot be inferred.

